# Liu2024 — Subject-Adaptive Calibration Transfer (S-JEPA × Riemannian)

**Why this experiment.** Cross-subject LOSO was at chance for every branch (riemann 52%,
sjepa 50%, fusion 53%): the discriminative covariance structure is subject-specific and does
not transfer — expected for acute-stroke MI. Meanwhile the decodable subgroup reaches ~75%
*within-subject*. The deployable question is therefore not "zero-shot transfer" but **how few
of a new subject's own trials are needed**, and whether cross-subject pretraining + S-JEPA help
get there faster.

**Design.** For each target subject and each calibration size `K`, compare on the *same* test
trials:
- **within_only** — head trained on the target's `K` calibration trials only (no transfer),
- **transfer_only** — head trained on the pooled other subjects (this is LOSO; `K`-independent),
- **transfer_plus_cal** — head trained on pooled others **+** the `K` calibration trials,

for branches **riemann** (tangent-space), **sjepa** (embeddings), **fusion** (both). Sweeping
`K` produces a **calibration curve** per strategy/branch.

**What a win looks like:** `transfer_plus_cal` beating `within_only` at small `K` means
cross-subject pretraining (and any S-JEPA contribution) genuinely reduces the calibration burden.
If the curves overlap, the honest conclusion is that only the subject's own trials matter and
neither transfer nor S-JEPA adds value — a clean result either way.

> **Honesty.** On the decodable subgroup this will likely reach ~70–75% at larger `K`, but that
> accuracy is driven by the within-subject Riemannian signal; the experiment's purpose is to
> measure whether transfer/S-JEPA *add* anything on top, not to attribute 70% to S-JEPA.

# 1. Setup

In [ ]:
import os, re, json, hashlib, random, builtins, platform, inspect
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
from scipy.io import loadmat
from scipy import signal
from scipy.linalg import eigh

from sklearn.metrics import balanced_accuracy_score, accuracy_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False; print(f"[setup] matplotlib unavailable: {exc}")

try:
    import torch; HAVE_TORCH = True
except Exception as exc:
    HAVE_TORCH = False; print(f"[setup] torch unavailable -> use cached embeddings: {exc}")

try:
    import mne; mne.set_log_level("WARNING"); HAVE_MNE = True
except Exception as exc:
    HAVE_MNE = False; print(f"[setup] mne unavailable: {exc}")

try:
    from braindecode.models import SignalJEPA_PreLocal; HAVE_BRAINDECODE = True
except Exception as exc:
    HAVE_BRAINDECODE = False; print(f"[setup] braindecode unavailable -> use cached embeddings: {exc}")

try:
    from pyriemann.estimation import Covariances; HAVE_PYRIEMANN = True
except Exception as exc:
    HAVE_PYRIEMANN = False; print(f"[setup] pyriemann unavailable -> numpy covariance fallback: {exc}")

os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
print("deps:", dict(torch=HAVE_TORCH, mne=HAVE_MNE, braindecode=HAVE_BRAINDECODE, pyriemann=HAVE_PYRIEMANN))


# 2. Configuration

## 2.1 Channels (carried)

In [ ]:
# Liu2024 source MAT channel conventions.
# Source files are organized as trials x 33 channels x samples:
#   0..29 = EEG-like channels, index 17 = CPz source reference,
#   30..31 = EOG, 32 = marker.
#
# The channel labels below follow the Liu2024 paper / EEGLAB location files.
# This matters for montage-dependent topomaps and any channel-position metadata.
SOURCE_EEG_CHANNEL_INDICES_30 = list(range(30))
SOURCE_REFERENCE_INDEX = 17
SOURCE_EEG_CHANNEL_INDICES_29 = [i for i in SOURCE_EEG_CHANNEL_INDICES_30 if i != SOURCE_REFERENCE_INDEX]

SOURCE_EEG_CHANNEL_NAMES_30 = [
    "Fp1", "Fp2", "Fz", "F3", "F4", "F7", "F8", "FCz", "FC3", "FC4",
    "FT7", "FT8", "Cz", "C3", "C4", "T3", "T4", "CPz",
    "CP3", "CP4", "TP7", "TP8", "Pz", "P3", "P4", "T5", "T6", "Oz", "O1", "O2",
]

# EOG and marker channels available in Liu2024 source MAT files.
SOURCE_EOG_CHANNEL_INDICES = [30, 31]
SOURCE_MARKER_CHANNEL_INDEX = 32


## 2.2 CONFIG (carried)

In [ ]:
WORKING_DIR = Path.cwd().resolve().parent.parent

CONFIG = {
    # ------------------------------------------------------------------
    # Paths / run identity
    # ------------------------------------------------------------------
    "artifact_dir": str(WORKING_DIR / "artifacts" / "liu2024-source-mat-sjepa-prelocal"),
    "source_extract_dir": str(WORKING_DIR / "liu2024_data" / "liu2024_figshare" / "sourcedata"),
    "experiment_name": "baseline_sjepa_prelocal",
    "config_note": "Clean MNE-style preprocessing pipeline builder + S-JEPA hyperparameter controls.",

    # ------------------------------------------------------------------
    # Dataset
    # ------------------------------------------------------------------
    "subjects_to_use": None,
    "exclude_subjects": [],
    "source_unit": "microvolts",
    "final_model_unit": "microvolts",

    # ------------------------------------------------------------------
    # Source-domain preprocessing before creating MNE RawArray
    # ------------------------------------------------------------------
    "demean_mode": "none",      # none, trial_mean, baseline_window_mean
    "baseline_window_s": [0.0, 2.0],
    "detrend_mode": "none",                     # none, constant, linear
    "eog_correction": "none",                   # none, linear_regression

    # Robust source-domain clipping / winsorization. Use cautiously.
    "artifact_clip_mode": "none",               # none, absolute, percentile
    "artifact_clip_abs_value": None,             # in source_unit, e.g. 150.0 when source_unit=microvolts
    "artifact_clip_percentile": 99.5,

    # ------------------------------------------------------------------
    # MNE Raw-level preprocessing
    # ------------------------------------------------------------------
    "reference_mode": "average",                # average, none
    "reference_timing": "before_resample_filter",  # before_resample_filter, after_resample_before_filter, after_filter
    "resample": True,
    "resample_sfreq": 128,

    "filter_enabled": True,
    "filter_low": 0.5,
    "filter_high": 40.0,
    "filter_method": "fir",                     # fir, iir
    "filter_phase": "zero",                     # zero, zero-double, minimum (FIR only)
    "filter_fir_design": "firwin",              # firwin, firwin2 (FIR only)
    "filter_l_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_h_trans_bandwidth": "auto",         # auto or float, FIR only
    "filter_iir_params": None,                    # example: {"order": 2, "ftype": "butter"}

    "notch_freqs": None,                          # example: [50.0]
    "notch_before_bandpass": False,

    # ------------------------------------------------------------------
    # Windowing and post-window cleaning
    # ------------------------------------------------------------------
    "target_window_s": 4.2,
    "target_window_samples": 537,
    "mi_window_start_s": 1.5,

    "reject_bad_trials": False,
    "reject_peak_to_peak_threshold": None,       # in final_model_unit
    "reject_abs_threshold": None,                # in final_model_unit
    "min_trials_per_class_after_reject": None,

    # ------------------------------------------------------------------
    # Fold-safe normalization. train_* modes are fit on each training split only.
    # ------------------------------------------------------------------
    "normalization_mode": "none",               # none, train_global_zscore, train_channel_zscore, train_channel_robust, trial_global_zscore, trial_channel_zscore
    "normalization_eps": 1e-6,

    # ------------------------------------------------------------------
    # Model / downstream strategy
    # ------------------------------------------------------------------
    "model_name": "SignalJEPA_PreLocal",
    "pretrained_mode": "from_pretrained",
    "pretrained_repo_id": "braindecode/signal-jepa_without-chans",
    "strategy": "new",                          # new, full
    "warmup_epochs": 10,

    # ------------------------------------------------------------------
    # Evaluation protocol
    # ------------------------------------------------------------------
    "evaluation_mode": "stratified_kfold",      # stratified_kfold, liu2024_repeated_60_40, repeated_stratified_split
    "cv_folds": 5,
    "n_repeats": 10,
    "test_size": 0.4,
    "split_random_state": 2026,
    "assert_balanced_folds": True,

    # ------------------------------------------------------------------
    # Training hyperparameters
    # ------------------------------------------------------------------
    "batch_size": 4,
    "n_epochs": 5000,
    "early_stopping_patience": 50,
    "val_split": 0.2,
    "learning_rate": 0.0003,
    "optimizer_name": "adam",                   # adam, adamw
    "weight_decay": 0.0,
    "gradient_clip_norm": None,
    "checkpoint_metric": "valid_loss",           # valid_loss, valid_balanced_accuracy
    "label_smoothing": 0.0,
    "prediction_balance_loss_weight": 1.0,

    # ------------------------------------------------------------------
    # braindecode on-the-fly augmentation.
    # Applied to the TRAINING iterator ONLY (via AugmentedDataLoader), so the
    # validation split skorch carves out internally is never augmented -> no leakage.
    # There is no fixed "number of augmented samples": the model sees a freshly
    # augmented view of the train fold every epoch. Control INTENSITY with each
    # transform's "probability" (how often it fires) and its magnitude params.
    # Ready-to-use configs are in the markdown cell just below CONFIG.
    # ------------------------------------------------------------------
    # "augmentation": {
    #     "enabled": False,        # master switch
    #     "name": "none",          # label, saved with artifacts
    #     "random_state": 2026,
    #     # each entry: {"name": <transform>, "probability": 0..1, <transform params>}
    #     "transforms": [],
    # },

    "augmentation": {
        "enabled": True,
        "name": "time_mask",
        "random_state": 2026,
        "transforms": [
        {
            "mask_len_samples": 64,
            "name": "smooth_time_mask",
            "probability": 0.5
        }
        ]
    },

    # ------------------------------------------------------------------
    # Reproducibility
    # ------------------------------------------------------------------
    "seed": 2026,
    "set_seed": True,
    "cv_random_state": 2026,
    "val_split_random_state": 2026,

    # ------------------------------------------------------------------
    # Diagnostics / interpretation
    # ------------------------------------------------------------------
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": False,
    "plot_individual_spatial_filters": False,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,
    "topomap_value_mode": "relative_zscore",    # raw, relative_zscore, relative_percent
    "topomap_cmap": "RdBu_r",
    "collapse_threshold": 0.9,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

## 2.3 Calibration-transfer settings

In [ ]:
CONFIG["experiment_name"] = "subject_adaptive_calibration_transfer"
CONFIG["artifact_dir"] = str(WORKING_DIR / "artifacts" / "liu2024-subject-adaptive-calibration")
CONFIG["config_note"] = "Pretrain cross-subject (EA) + per-subject calibration; calibration curve per strategy/branch."
CONFIG["augmentation"] = {"enabled": False, "name": "none", "random_state": 2026, "transforms": []}

CONFIG["adapt"] = {
    # genuine within-subject decodable subgroup (uncorrected honest p<0.05 OR FgMDM>=0.70); or "all"
    "target_subjects": [7, 22, 23, 28, 40, 44],
    "branches": ["riemann", "sjepa", "fusion"],
    "strategies": ["within_only", "transfer_only", "transfer_plus_cal"],
    "calibration_grid": [0, 4, 8, 12, 16, 24],   # K = #target trials used to calibrate
    "n_repeats": 10,                              # random calibration draws per K
    "euclidean_alignment": True,
    "cov_estimator": "oas",
    "logreg_C": 1.0,
    "standardize": True,
    "calibration_upweight": 1.0,                  # >1 upweights calibration trials in transfer_plus_cal
    # S-JEPA embeddings: point this at the cached file from the fusion run; else live-extract.
    "sjepa_embeddings_path": None,
    "embedding_module": "final_layer", "embedding_pool": "mean", "embedding_batch": 64,
    "align_before_sjepa": True,
    "seed": 2026,
}
print("Targets:", CONFIG["adapt"]["target_subjects"])
print("Calibration grid (K):", CONFIG["adapt"]["calibration_grid"], "| strategies:", CONFIG["adapt"]["strategies"])


## 2.4 Constants / artifacts / reproducibility (carried)

In [ ]:
# Liu2024 source MAT constants.
LIU_SOURCE_SFREQ = 500
LIU_EXPECTED_TRIALS_PER_SUBJECT = 40
LIU_EXPECTED_SOURCE_CHANNELS = 33
LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL = 4000

# Liu source MAT files include EEG + EOG + marker channels.
# Keep the 29 EEG channels used in the Liu paper baseline and drop CPz because it is the source reference channel.
EEG_CHANNEL_INDICES = SOURCE_EEG_CHANNEL_INDICES_29
EEG_CHANNEL_NAMES = [
    name for idx, name in enumerate(SOURCE_EEG_CHANNEL_NAMES_30)
    if idx != SOURCE_REFERENCE_INDEX
]

SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
TARGET_N_CLASSES = 2

if bool(CONFIG.get("resample", True)):
    EFFECTIVE_SFREQ = float(CONFIG.get("resample_sfreq", 128))
else:
    EFFECTIVE_SFREQ = float(LIU_SOURCE_SFREQ)

CONFIG["effective_sfreq"] = EFFECTIVE_SFREQ
CONFIG["sfreq"] = EFFECTIVE_SFREQ  # compatibility with existing cells/artifacts

if CONFIG.get("target_window_samples", None) is None:
    WINDOW_SAMPLES = int(round(float(CONFIG["target_window_s"]) * EFFECTIVE_SFREQ))
else:
    WINDOW_SAMPLES = int(CONFIG["target_window_samples"])

TARGET_TRIAL_DURATION_S = WINDOW_SAMPLES / EFFECTIVE_SFREQ
MI_WINDOW_START_SAMPLE = int(round(float(CONFIG["mi_window_start_s"]) * EFFECTIVE_SFREQ))
MI_WINDOW_STOP_SAMPLE = MI_WINDOW_START_SAMPLE + WINDOW_SAMPLES

PREPROCESSING_KEYS = [
    "source_unit", "final_model_unit",
    "demean_mode", "baseline_window_s", "detrend_mode", "eog_correction",
    "artifact_clip_mode", "artifact_clip_abs_value", "artifact_clip_percentile",
    "reference_mode", "reference_timing", "resample", "resample_sfreq", "effective_sfreq",
    "filter_enabled", "filter_low", "filter_high", "filter_method", "filter_phase",
    "filter_fir_design", "filter_l_trans_bandwidth", "filter_h_trans_bandwidth", "filter_iir_params",
    "notch_freqs", "notch_before_bandpass",
    "mi_window_start_s", "target_window_s", "target_window_samples",
    "reject_bad_trials", "reject_peak_to_peak_threshold", "reject_abs_threshold",
    "normalization_mode", "normalization_eps",
]

TRAINING_KEYS = [
    "strategy", "batch_size", "learning_rate", "optimizer_name", "weight_decay",
    "val_split", "early_stopping_patience", "n_epochs",
    "augmentation",
]

EVALUATION_KEYS = [
    "evaluation_mode", "cv_folds", "n_repeats", "test_size",
    "cv_random_state", "split_random_state", "val_split_random_state",
]

def summarize_selected_config(keys):
    return {k: CONFIG.get(k) for k in keys}

PREPROCESSING_CONFIG = summarize_selected_config(PREPROCESSING_KEYS)
TRAINING_CONFIG = summarize_selected_config(TRAINING_KEYS)
EVALUATION_CONFIG = summarize_selected_config(EVALUATION_KEYS)

def print_config_block(title, values):
    print(title)
    for key, value in values.items():
        print(f"  {key:34s}: {value}")

print("Effective Liu2024 Source MAT settings:")
print(f"  Experiment:                        {CONFIG.get('experiment_name')}")
print(f"  Note:                              {CONFIG.get('config_note')}")
print(f"  Channels:                          {len(EEG_CHANNEL_NAMES)}")
print(f"  Channel names:                     {EEG_CHANNEL_NAMES}")
print(f"  Source sfreq:                      {LIU_SOURCE_SFREQ} Hz")
print(f"  Effective sfreq:                   {EFFECTIVE_SFREQ} Hz")
print(f"  MI window start / samples:         {CONFIG['mi_window_start_s']} s / {WINDOW_SAMPLES}")
print(f"  Effective window duration:         {TARGET_TRIAL_DURATION_S:.4f} s")
print(f"  Evaluation mode:                   {CONFIG.get('evaluation_mode')}")
print(f"  Fixed seed:                        base={CONFIG.get('seed')} | cv={CONFIG.get('cv_random_state')} | split={CONFIG.get('split_random_state')} | val={CONFIG.get('val_split_random_state')}")
print_config_block("\nPreprocessing config:", PREPROCESSING_CONFIG)
print_config_block("\nTraining config:", TRAINING_CONFIG)
print_config_block("\nEvaluation config:", EVALUATION_CONFIG)


In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2)

print(f"Run ID:     {RUN_ID}")
print(f"Artifacts:  {ARTIFACT_DIR}")
print(f"Config:     {config_path}")


In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"])
if CONFIG["set_seed"]:
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")


# 3. Carried machinery (verbatim)

### 3.1 Data-loading helpers

In [ ]:
def find_source_mat_files(root):
    root = Path(root)
    return sorted(root.rglob("*.mat")) if root.exists() else []

def subject_id_from_path(path):
    s = str(path)
    m = re.search(r"sub[-_ ]?(\d{1,2})", s, flags=re.IGNORECASE)
    if m:
        return int(m.group(1))
    nums = re.findall(r"\d+", Path(path).stem)
    if nums:
        return int(nums[-1])
    raise ValueError(f"Could not infer subject id from path: {path}")

def _is_mat_struct(x):
    return hasattr(x, "_fieldnames")

def _walk_mat_object(obj, prefix=""):
    """Recursively walk scipy-loaded MATLAB dicts/structs.

    Liu2024 source files may expose only a top-level `eeg` object instead of
    top-level `rawdata` and `labels`. This walker lets the loader find nested
    arrays without assuming one exact MATLAB struct layout.
    """
    if isinstance(obj, dict):
        for k, v in obj.items():
            if str(k).startswith("__"):
                continue
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif _is_mat_struct(obj):
        for k in obj._fieldnames:
            v = getattr(obj, k)
            name = f"{prefix}.{k}" if prefix else str(k)
            yield name, v
            yield from _walk_mat_object(v, name)
    elif isinstance(obj, np.ndarray):
        if obj.dtype == object and obj.size == 1:
            yield from _walk_mat_object(obj.item(), prefix)
        elif obj.dtype == object:
            for idx, item in np.ndenumerate(obj):
                yield from _walk_mat_object(item, f"{prefix}{idx}")

def mat_structure_preview(path, max_rows=200):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)
    rows = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray):
            rows.append({
                "name": name,
                "type": "ndarray",
                "shape": str(value.shape),
                "dtype": str(value.dtype),
            })
        else:
            rows.append({
                "name": name,
                "type": type(value).__name__,
                "shape": "",
                "dtype": "",
            })
    return pd.DataFrame(rows).head(max_rows)

def _normalize_rawdata_shape(rawdata, labels=None):
    arr = np.asarray(rawdata)
    if arr.ndim != 3:
        raise ValueError(f"Expected 3D rawdata, got shape={arr.shape}")

    # Prefer the label-count axis as the trial axis when labels are available.
    if labels is not None:
        n_labels = int(np.asarray(labels).size)
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size == n_labels]
    else:
        trial_axes = []

    if not trial_axes:
        trial_axes = [axis for axis, size in enumerate(arr.shape) if size in (39, 40)]

    if trial_axes and trial_axes[0] != 0:
        arr = np.moveaxis(arr, trial_axes[0], 0)

    # After trial-axis normalization, the time axis should be the largest axis.
    time_axis = int(np.argmax(arr.shape[1:]) + 1)
    if time_axis != 2:
        arr = np.moveaxis(arr, time_axis, 2)

    if arr.shape[1] < 29 or arr.shape[2] < 1000:
        raise ValueError(f"Could not normalize rawdata to trials x channels x samples, got {arr.shape}")
    return arr

def _score_raw_candidate(name, arr):
    lname = name.lower()
    score = 0
    if "rawdata" in lname or "raw" in lname or "data" in lname:
        score += 10
    if arr.ndim == 3:
        score += 5
    if any(size in (39, 40) for size in arr.shape):
        score += 3
    if max(arr.shape) >= 3000:
        score += 2
    if "eeg" in lname:
        score += 1
    return score

def _score_label_candidate(name, arr):
    lname = name.lower()
    flat = np.asarray(arr).ravel()
    unique = set(np.unique(flat).astype(str).tolist()) if flat.size <= 200 else set()
    score = 0
    if "label" in lname or "class" in lname or lname.split(".")[-1] in {"y", "labels"}:
        score += 10
    if flat.size in (39, 40):
        score += 3
    if unique and unique.issubset({"0", "1", "2"}):
        score += 2
    return score

def validate_liu_source_subject(rawdata, labels, subject_id, path=None):
    """Validate the fixed Liu source MAT layout assumptions."""
    expected_trials = LIU_EXPECTED_TRIALS_PER_SUBJECT
    expected_channels = LIU_EXPECTED_SOURCE_CHANNELS
    expected_samples = LIU_EXPECTED_SOURCE_SAMPLES_PER_TRIAL

    if rawdata.shape[0] != labels.size:
        raise ValueError(
            f"Subject {subject_id}: labels/trials mismatch. "
            f"rawdata={rawdata.shape}, labels={labels.shape}, path={path}"
        )
    if rawdata.shape[0] != expected_trials:
        print(f"WARNING subject {subject_id}: expected {expected_trials} trials, got {rawdata.shape[0]}")
    if rawdata.shape[1] < len(SOURCE_EEG_CHANNEL_INDICES_30):
        raise ValueError(f"Subject {subject_id}: expected at least 30 EEG-like channels, got {rawdata.shape}")
    if rawdata.shape[1] != expected_channels:
        print(f"WARNING subject {subject_id}: expected {expected_channels} source channels, got {rawdata.shape[1]}")
    if rawdata.shape[2] != expected_samples:
        print(f"WARNING subject {subject_id}: expected {expected_samples} samples/trial, got {rawdata.shape[2]}")

    unique = set(np.unique(labels).astype(int).tolist())
    if not unique.issubset({0, 1, 2}):
        raise ValueError(f"Subject {subject_id}: unexpected labels {sorted(unique)}")

    y0 = labels_to_zero_based(labels)
    counts = np.bincount(y0, minlength=TARGET_N_CLASSES)
    if counts.min() == 0:
        raise ValueError(f"Subject {subject_id}: missing class after zero-based conversion, counts={counts.tolist()}")
    if counts[0] != counts[1]:
        print(f"WARNING subject {subject_id}: class counts are not balanced: {counts.tolist()}")

def load_subject_mat(path):
    mat = loadmat(path, squeeze_me=True, struct_as_record=False)

    arrays = []
    for name, value in _walk_mat_object(mat):
        if isinstance(value, np.ndarray) and value.dtype != object:
            arrays.append((name, np.asarray(value)))

    raw_candidates, label_candidates = [], []
    for name, arr in arrays:
        if arr.ndim == 3:
            raw_candidates.append((_score_raw_candidate(name, arr), name, arr))
        elif arr.ndim in (1, 2):
            label_candidates.append((_score_label_candidate(name, arr), name, arr))

    if not raw_candidates or not label_candidates:
        preview = mat_structure_preview(path)
        preview_path = ARTIFACT_DIR / f"mat_structure_failure_{Path(path).stem}.csv"
        preview.to_csv(preview_path, index=False)
        print(f"MAT structure preview for failure saved to: {preview_path}")
        display(preview.head(40))
        raise KeyError(f"Could not locate 3D raw data and labels in {path}")

    _, raw_name, raw_arr = sorted(raw_candidates, key=lambda x: x[0], reverse=True)[0]
    _, label_name, label_arr = sorted(label_candidates, key=lambda x: x[0], reverse=True)[0]

    labels = np.asarray(label_arr).astype(int).ravel()
    rawdata = _normalize_rawdata_shape(raw_arr, labels=labels).astype(np.float64)

    if labels.size != rawdata.shape[0]:
        raise ValueError(f"Label count mismatch in {path}: labels={labels.shape}, rawdata={rawdata.shape}")

    return rawdata, labels.astype(int), raw_name, label_name


### 3.2 Preprocessing pipeline

In [ ]:
def make_liu_info(sfreq):
    info = mne.create_info(
        ch_names=EEG_CHANNEL_NAMES,
        sfreq=float(sfreq),
        ch_types=["eeg"] * len(EEG_CHANNEL_NAMES),  # type: ignore
    )
    montage = mne.channels.make_standard_montage("standard_1020")
    info.set_montage(montage, match_case=False, on_missing="ignore")
    return info

def labels_to_zero_based(labels):
    labels = np.asarray(labels).astype(int).ravel()
    unique = set(np.unique(labels).tolist())
    if unique.issubset({1, 2}):
        return labels - 1
    if unique.issubset({0, 1}):
        return labels
    raise ValueError(f"Unexpected labels: {sorted(unique)}")

def source_values_to_mne_volts(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("source_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e-6
    if unit in ("mv", "millivolt", "millivolts"):
        return arr * 1e-3
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported source_unit={config.get('source_unit')}")

def mne_volts_to_model_unit(data, config=None):
    config = CONFIG if config is None else config
    arr = np.asarray(data, dtype=np.float64)
    unit = str(config.get("final_model_unit", "microvolts")).lower()
    if unit in ("uv", "microvolt", "microvolts"):
        return arr * 1e6
    if unit in ("v", "volt", "volts"):
        return arr
    raise ValueError(f"Unsupported final_model_unit={config.get('final_model_unit')}")

def _none_like(value):
    return value is None or str(value).lower() in ("none", "off", "false", "")

def build_preprocessing_pipeline(config):
    """Return a readable pipeline plan.

    The pipeline is represented as a list of dictionaries rather than hidden global logic.
    Every row is logged and saved in the run metadata through the preprocessing step list.
    """
    pipeline = []

    # Fixed source structure.
    pipeline.append({
        "stage": "source",
        "name": "select_eeg_channels",
        "description": "select Liu EEG channels, drop CPz reference, EOG, and marker before model input",
        "enabled": True,
    })

    pipeline.append({
        "stage": "source",
        "name": "demean",
        "mode": config.get("demean_mode", "none"),
        "baseline_window_s": config.get("baseline_window_s"),
        "enabled": not _none_like(config.get("demean_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "detrend",
        "mode": config.get("detrend_mode", "none"),
        "enabled": not _none_like(config.get("detrend_mode", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "eog_correction",
        "mode": config.get("eog_correction", "none"),
        "enabled": not _none_like(config.get("eog_correction", "none")),
    })

    pipeline.append({
        "stage": "source",
        "name": "artifact_clipping",
        "mode": config.get("artifact_clip_mode", "none"),
        "abs_value": config.get("artifact_clip_abs_value"),
        "percentile": config.get("artifact_clip_percentile"),
        "enabled": not _none_like(config.get("artifact_clip_mode", "none")),
    })

    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()
    if reference_timing == "before_resample_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "before_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({"stage": "mne_raw", "name": "resample", "sfreq": config.get("resample_sfreq"), "enabled": bool(config.get("resample", True))})

    if reference_timing == "after_resample_before_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    pipeline.append({
        "stage": "mne_raw",
        "name": "bandpass_filter",
        "enabled": bool(config.get("filter_enabled", True)),
        "l_freq": config.get("filter_low"),
        "h_freq": config.get("filter_high"),
        "method": config.get("filter_method"),
        "phase": config.get("filter_phase"),
        "fir_design": config.get("filter_fir_design"),
        "iir_params": config.get("filter_iir_params"),
    })

    if reference_timing == "after_filter":
        pipeline.append({"stage": "mne_raw", "name": "reference", "timing": reference_timing, "mode": config.get("reference_mode"), "enabled": not _none_like(config.get("reference_mode"))})

    if not bool(config.get("notch_before_bandpass", False)):
        pipeline.append({"stage": "mne_raw", "name": "notch_filter", "timing": "after_bandpass", "freqs": config.get("notch_freqs"), "enabled": not _none_like(config.get("notch_freqs"))})

    pipeline.append({
        "stage": "window",
        "name": "crop_fixed_mi_window",
        "start_s": config.get("mi_window_start_s"),
        "target_window_samples": config.get("target_window_samples"),
        "enabled": True,
    })

    pipeline.append({
        "stage": "window",
        "name": "bad_trial_rejection",
        "enabled": bool(config.get("reject_bad_trials", False)),
        "peak_to_peak_threshold": config.get("reject_peak_to_peak_threshold"),
        "abs_threshold": config.get("reject_abs_threshold"),
    })

    pipeline.append({
        "stage": "split",
        "name": "fold_safe_normalization",
        "mode": config.get("normalization_mode", "none"),
        "enabled": not _none_like(config.get("normalization_mode", "none")),
    })

    return pipeline

def describe_pipeline(pipeline):
    lines = []
    for step in pipeline:
        status = "ON" if step.get("enabled", False) else "off"
        parts = [f"[{status}] {step.get('stage')}::{step.get('name')}"]
        for key, value in step.items():
            if key not in ("stage", "name", "description", "enabled") and value is not None:
                parts.append(f"{key}={value}")
        if step.get("description"):
            parts.append(f"- {step['description']}")
        lines.append(" | ".join(parts))
    return lines

PREPROCESSING_PIPELINE = build_preprocessing_pipeline(CONFIG)
print("Configured preprocessing pipeline:")
for line in describe_pipeline(PREPROCESSING_PIPELINE):
    print("  - " + line)

def apply_source_demean(X_eeg, subject_id, config, steps):
    mode = str(config.get("demean_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source demean skipped")
        return X

    if mode == "trial_mean":
        steps.append("source demean: subtract trial/channel mean over time")
        return X - X.mean(axis=-1, keepdims=True)

    if mode == "baseline_window_mean":
        baseline = config.get("baseline_window_s", [0.0, 2.0])
        if baseline is None or len(baseline) != 2:
            raise ValueError("baseline_window_s must be [start_s, stop_s] for baseline_window_mean.")
        start_s, stop_s = float(baseline[0]), float(baseline[1])
        start = int(round(start_s * LIU_SOURCE_SFREQ))
        stop = int(round(stop_s * LIU_SOURCE_SFREQ))
        if start < 0 or stop <= start or stop > X.shape[-1]:
            raise ValueError(f"Subject {subject_id}: invalid baseline_window_s={baseline} for source length {X.shape[-1]}")
        steps.append(f"source demean: subtract baseline mean {baseline}s")
        return X - X[:, :, start:stop].mean(axis=-1, keepdims=True)

    raise ValueError(f"Unsupported demean_mode={config.get('demean_mode')}")

def apply_source_detrend(X_eeg, subject_id, config, steps):
    mode = str(config.get("detrend_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)
    if mode in ("none", "off", "false"):
        steps.append("source detrend skipped")
        return X
    if mode == "constant":
        steps.append("source detrend: scipy.signal.detrend(type='constant')")
        return signal.detrend(X, axis=-1, type="constant")
    if mode == "linear":
        steps.append("source detrend: scipy.signal.detrend(type='linear')")
        return signal.detrend(X, axis=-1, type="linear")
    raise ValueError(f"Unsupported detrend_mode={config.get('detrend_mode')}")

def apply_eog_correction(X_eeg, rawdata, subject_id, config, steps):
    mode = str(config.get("eog_correction", "none")).lower()
    if mode in ("none", "off", "false"):
        steps.append("EOG correction skipped")
        return X_eeg

    if mode != "linear_regression":
        raise ValueError(
            "Only eog_correction='linear_regression' is implemented in this source-MAT notebook. "
            "ICA is intentionally not included because the source pipeline drops EOG before model input and "
            "the dataset has only 40 trials per subject."
        )

    if rawdata.shape[1] <= max(SOURCE_EOG_CHANNEL_INDICES):
        raise ValueError(f"Subject {subject_id}: rawdata does not contain expected EOG channels.")

    X = np.asarray(X_eeg, dtype=np.float64)
    eog = np.asarray(rawdata[:, SOURCE_EOG_CHANNEL_INDICES, :], dtype=np.float64)

    n_trials, n_chans, n_samples = X.shape
    eog_2d = eog.transpose(0, 2, 1).reshape(-1, len(SOURCE_EOG_CHANNEL_INDICES))
    eeg_2d = X.transpose(0, 2, 1).reshape(-1, n_chans)

    design = np.column_stack([np.ones(eog_2d.shape[0]), eog_2d])
    beta, *_ = np.linalg.lstsq(design, eeg_2d, rcond=None)
    eog_contribution = design[:, 1:] @ beta[1:, :]
    corrected = eeg_2d - eog_contribution
    corrected = corrected.reshape(n_trials, n_samples, n_chans).transpose(0, 2, 1)

    steps.append("EOG correction: linear regression using HEOG/VEOG before dropping EOG")
    return corrected

def apply_source_artifact_clipping(X_eeg, subject_id, config, steps):
    mode = str(config.get("artifact_clip_mode", "none")).lower()
    X = np.asarray(X_eeg, dtype=np.float64)

    if mode in ("none", "off", "false"):
        steps.append("source artifact clipping skipped")
        return X, {"artifact_clip_applied": False, "artifact_clip_threshold": None}

    if mode == "absolute":
        threshold = config.get("artifact_clip_abs_value")
        if threshold is None:
            raise ValueError("artifact_clip_abs_value must be set when artifact_clip_mode='absolute'.")
        threshold = float(threshold)
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: absolute ±{threshold:g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    if mode == "percentile":
        pct = float(config.get("artifact_clip_percentile", 99.5))
        threshold = float(np.nanpercentile(np.abs(X), pct))
        clipped = np.clip(X, -threshold, threshold)
        changed = int(np.sum(clipped != X))
        steps.append(f"source artifact clipping: percentile {pct:g}% -> ±{threshold:.4g} {config.get('source_unit')} | changed_values={changed}")
        return clipped, {
            "artifact_clip_applied": True,
            "artifact_clip_mode": mode,
            "artifact_clip_percentile": pct,
            "artifact_clip_threshold": threshold,
            "artifact_clip_changed_values": changed,
        }

    raise ValueError(f"Unsupported artifact_clip_mode={config.get('artifact_clip_mode')}")

def apply_reference(raw, config, steps, timing_label):
    mode = str(config.get("reference_mode", "average")).lower()
    if mode in ("none", "off", "false"):
        steps.append(f"reference skipped at {timing_label}")
        return raw
    if mode == "average":
        raw.set_eeg_reference("average", projection=False, verbose=False)
        steps.append(f"average reference at {timing_label}")
        return raw
    raise ValueError(f"Unsupported reference_mode={config.get('reference_mode')}")

def apply_notch(raw, config, steps, timing_label):
    freqs = config.get("notch_freqs", None)
    if freqs is None or freqs == []:
        return raw
    raw.notch_filter(freqs=freqs, verbose=False)
    steps.append(f"notch filter {freqs} Hz at {timing_label}")
    return raw

def apply_resample(raw, config, steps):
    if bool(config.get("resample", True)):
        target = float(config.get("resample_sfreq", 128))
        raw.resample(target, verbose=False)
        steps.append(f"resample to {target:g} Hz")
    else:
        steps.append("resample skipped; kept source 500 Hz")
    return raw

def _float_or_none_or_auto(value):
    if value is None:
        return None
    if isinstance(value, str) and value.lower() == "auto":
        return "auto"
    return float(value)

def apply_bandpass(raw, config, steps):
    if not bool(config.get("filter_enabled", True)):
        steps.append("bandpass skipped")
        return raw

    l_freq = config.get("filter_low", None)
    h_freq = config.get("filter_high", None)
    l_freq = None if l_freq is None else float(l_freq)
    h_freq = None if h_freq is None else float(h_freq)

    method = str(config.get("filter_method", "fir")).lower()
    if method == "iir":
        iir_params = config.get("filter_iir_params", None)
        if iir_params is None:
            iir_params = {"order": 2, "ftype": "butter"}
        raw.filter(l_freq=l_freq, h_freq=h_freq, method="iir", iir_params=iir_params, verbose=False)
        steps.append(f"IIR bandpass {l_freq}–{h_freq} Hz | params={iir_params}")
    elif method == "fir":
        raw.filter(
            l_freq=l_freq,
            h_freq=h_freq,
            method="fir",
            phase=config.get("filter_phase", "zero"),
            fir_design=config.get("filter_fir_design", "firwin"),
            l_trans_bandwidth=_float_or_none_or_auto(config.get("filter_l_trans_bandwidth", "auto")),
            h_trans_bandwidth=_float_or_none_or_auto(config.get("filter_h_trans_bandwidth", "auto")),
            verbose=False,
        )
        steps.append(
            f"FIR bandpass {l_freq}–{h_freq} Hz | phase={config.get('filter_phase')} | "
            f"fir_design={config.get('filter_fir_design')}"
        )
    else:
        raise ValueError(f"Unsupported filter_method={config.get('filter_method')}")
    return raw

def apply_mne_raw_pipeline(raw, config, steps):
    reference_timing = str(config.get("reference_timing", "before_resample_filter")).lower()

    if reference_timing == "before_resample_filter":
        raw = apply_reference(raw, config, steps, "before resample/filter")

    if bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "before resample/filter")

    raw = apply_resample(raw, config, steps)

    if reference_timing == "after_resample_before_filter":
        raw = apply_reference(raw, config, steps, "after resample before filter")

    raw = apply_bandpass(raw, config, steps)

    if reference_timing == "after_filter":
        raw = apply_reference(raw, config, steps, "after filter")

    if not bool(config.get("notch_before_bandpass", False)):
        raw = apply_notch(raw, config, steps, "after bandpass")

    return raw

def maybe_reject_bad_trials(X_win, y, subject_id, config, steps):
    stats = {
        "reject_bad_trials": bool(config.get("reject_bad_trials", False)),
        "n_trials_before_reject": int(len(y)),
        "n_trials_after_reject": int(len(y)),
        "n_rejected_trials": 0,
        "rejection_skipped": False,
    }

    if not bool(config.get("reject_bad_trials", False)):
        steps.append("bad-trial rejection skipped")
        return X_win, y, stats

    keep = np.ones(len(y), dtype=bool)

    ptp_threshold = config.get("reject_peak_to_peak_threshold", None)
    if ptp_threshold is not None:
        ptp = np.ptp(X_win, axis=-1).max(axis=1)
        keep &= ptp <= float(ptp_threshold)
        stats["reject_peak_to_peak_threshold"] = float(ptp_threshold)
        stats["max_trial_peak_to_peak"] = float(np.max(ptp))

    abs_threshold = config.get("reject_abs_threshold", None)
    if abs_threshold is not None:
        max_abs = np.max(np.abs(X_win), axis=(1, 2))
        keep &= max_abs <= float(abs_threshold)
        stats["reject_abs_threshold"] = float(abs_threshold)
        stats["max_trial_abs"] = float(np.max(max_abs))

    proposed_y = y[keep]
    min_required = config.get("min_trials_per_class_after_reject", None)
    if min_required is None:
        if config.get("evaluation_mode") == "liu2024_repeated_60_40":
            min_required = 12
        else:
            min_required = int(config.get("cv_folds", 5))
    proposed_counts = np.bincount(proposed_y, minlength=TARGET_N_CLASSES)

    if len(proposed_y) == 0 or proposed_counts.min() < int(min_required):
        steps.append(
            "bad-trial rejection skipped because it would leave too few samples "
            f"per class: proposed_counts={proposed_counts.tolist()}, min_required={min_required}"
        )
        stats["rejection_skipped"] = True
        stats["proposed_class_counts_after_reject"] = proposed_counts.tolist()
        return X_win, y, stats

    X_new = X_win[keep]
    y_new = proposed_y
    stats["n_trials_after_reject"] = int(len(y_new))
    stats["n_rejected_trials"] = int(np.sum(~keep))
    stats["class_counts_after_reject"] = np.bincount(y_new, minlength=TARGET_N_CLASSES).tolist()
    steps.append(
        f"bad-trial rejection applied: rejected={stats['n_rejected_trials']} / {stats['n_trials_before_reject']} | "
        f"class_counts={stats['class_counts_after_reject']}"
    )
    return X_new, y_new, stats

def preprocess_subject_configurable(rawdata, labels, subject_id, config=None):
    """Apply the configured preprocessing pipeline to one Liu2024 source subject.

    Order:
      1. source-domain operations: channel selection, mean removal, detrending, EOG regression, clipping
      2. MNE RawArray operations: reference, notch, resample, bandpass
      3. window crop and optional trial rejection
      4. fold-safe normalization later inside training split code
    """
    config = CONFIG if config is None else config

    if rawdata.ndim != 3:
        raise ValueError(f"Subject {subject_id}: expected 3D rawdata, got {rawdata.shape}")

    n_trials = rawdata.shape[0]
    steps = describe_pipeline(build_preprocessing_pipeline(config))
    runtime_steps = []
    preprocessing_stats = {}

    X_eeg = rawdata[:, EEG_CHANNEL_INDICES, :].astype(np.float64)
    runtime_steps.append("select 29 EEG channels; drop CPz source reference, EOG, and marker")

    X_eeg = apply_source_demean(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_source_detrend(X_eeg, subject_id, config, runtime_steps)
    X_eeg = apply_eog_correction(X_eeg, rawdata, subject_id, config, runtime_steps)
    X_eeg, clip_stats = apply_source_artifact_clipping(X_eeg, subject_id, config, runtime_steps)
    preprocessing_stats.update(clip_stats)

    X_eeg_volts = source_values_to_mne_volts(X_eeg, config)
    runtime_steps.append(f"convert source {config.get('source_unit')} to MNE volts")

    continuous = X_eeg_volts.transpose(1, 0, 2).reshape(len(EEG_CHANNEL_INDICES), -1)
    info = make_liu_info(LIU_SOURCE_SFREQ)
    raw = mne.io.RawArray(continuous, info, verbose=False)

    raw = apply_mne_raw_pipeline(raw, config, runtime_steps)

    effective_sfreq = float(raw.info["sfreq"])
    data = mne_volts_to_model_unit(raw.get_data(), config)
    runtime_steps.append(f"convert MNE volts to model {config.get('final_model_unit')}")

    expected_samples_per_trial = int(round(rawdata.shape[2] * effective_sfreq / float(LIU_SOURCE_SFREQ)))
    total_expected = n_trials * expected_samples_per_trial
    if data.shape[1] != total_expected:  # type: ignore
        n_full = data.shape[1] // n_trials  # type: ignore
        expected_samples_per_trial = n_full
        data = data[:, :n_trials * expected_samples_per_trial]  # type: ignore
        runtime_steps.append(f"trim continuous samples to full trials: {expected_samples_per_trial} samples/trial")

    X_rs = data.reshape(len(EEG_CHANNEL_INDICES), n_trials, expected_samples_per_trial).transpose(1, 0, 2)  # type: ignore

    start_sample = int(round(float(config["mi_window_start_s"]) * effective_sfreq))
    window_samples = int(config["target_window_samples"]) if config.get("target_window_samples") is not None else int(round(float(config["target_window_s"]) * effective_sfreq))
    stop_sample = start_sample + window_samples

    if stop_sample > X_rs.shape[-1]:
        raise ValueError(
            f"Subject {subject_id}: crop [{start_sample}:{stop_sample}] exceeds trial length "
            f"{X_rs.shape[-1]} at effective_sfreq={effective_sfreq}"
        )

    X_win = X_rs[:, :, start_sample:stop_sample]
    runtime_steps.append(f"crop fixed window samples [{start_sample}:{stop_sample}]")

    y = labels_to_zero_based(labels)
    X_win, y, reject_stats = maybe_reject_bad_trials(X_win, y, subject_id, config, runtime_steps)
    preprocessing_stats.update(reject_stats)

    # Save both the intended pipeline and the actual runtime steps.
    preprocessing_stats["pipeline_plan"] = steps
    preprocessing_stats["runtime_steps"] = runtime_steps

    return X_win.astype(np.float32), y.astype(np.int64), int(expected_samples_per_trial), runtime_steps, preprocessing_stats


### 3.3 Dataset classes

In [ ]:
class SubjectArrayDataset(Dataset):
    def __init__(self, X, y, subject_id):
        self.X = np.asarray(X, dtype=np.float32)
        self.y = np.asarray(y, dtype=np.int64)
        self.subject_id = str(subject_id)
        if self.X.ndim != 3:
            raise ValueError(f"SubjectArrayDataset expects X as N x C x T, got shape={self.X.shape}.")
        if len(self.X) != len(self.y):
            raise ValueError(f"X/y length mismatch: {len(self.X)} windows vs {len(self.y)} labels.")

    def __len__(self):
        return int(len(self.y))

    def __getitem__(self, idx):
        x = np.asarray(self.X[idx], dtype=np.float32)
        if x.ndim != 2:
            raise ValueError(f"Expected one EEG window as C x T, got shape={x.shape}.")
        return x, int(self.y[idx])


class FoldNormalizedDataset(Dataset):
    """Wrap a dataset and apply either fold-fitted or trial-wise normalization."""

    def __init__(self, dataset, normalizer_state):
        self.dataset = dataset
        self.normalizer_state = normalizer_state or {"mode": "none"}

    def __len__(self):
        return len(self.dataset)

    def __getitem__(self, idx):
        x, y = self.dataset[idx]
        x = apply_normalizer_to_array(x, self.normalizer_state)
        if x.ndim != 2:
            raise ValueError(f"Normalization must return C x T, got shape={x.shape}.")
        return x.astype(np.float32), int(y)


### 3.4 JSON helpers

In [ ]:
def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()


### 3.5 Pretrained S-JEPA builder (only used if embeddings must be extracted live)

In [ ]:
NEW_LAYER_PREFIXES = ("spatial_conv.", "final_layer.")

def build_model():
    common_kwargs = {
        "n_chans": len(CH_NAMES),
        "chs_info": CHS_INFO,
        "n_times": WINDOW_SAMPLES,
        "n_outputs": TARGET_N_CLASSES,
    }
    mode = CONFIG["pretrained_mode"]
    if mode == "from_pretrained":
        model = SignalJEPA_PreLocal.from_pretrained(
            CONFIG["pretrained_repo_id"],
            **common_kwargs,
            strict=False,
        )
        info = {
            "loading_path": "from_pretrained",
            "repo_id": CONFIG["pretrained_repo_id"],
            "mode": mode,
        }
    elif mode == "random":
        model = SignalJEPA_PreLocal(**common_kwargs)
        info = {
            "loading_path": "random_initialization",
            "repo_id": None,
            "mode": mode,
        }
    else:
        raise ValueError("pretrained_mode must be 'from_pretrained' or 'random'.")
    info["model_name"] = CONFIG["model_name"]
    return model, info

def set_trainable_params_for_phase(model, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")

    if phase == "full":
        for _, p in model.named_parameters():
            p.requires_grad = True
        phase_groups = ["all_parameters"]
    else:
        for _, p in model.named_parameters():
            p.requires_grad = False
        for name, p in model.named_parameters():
            if any(name.startswith(prefix) for prefix in NEW_LAYER_PREFIXES):
                p.requires_grad = True
        phase_groups = list(NEW_LAYER_PREFIXES)

    trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters for phase={phase}.")
    return {
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def summarize_trainable_parameters(model):
    rows = []
    for name, param in model.named_parameters():
        if param.requires_grad:
            rows.append({
                "name": name,
                "numel": int(param.numel()),
                "shape": list(param.shape),
            })
    return rows

def count_trainable_from_rows(rows):
    return int(sum(row.get("numel", 0) for row in rows))


### 3.6 Data driver — X_ALL / Y_ALL / SUBJECT_ID_ALL / CH_NAMES / CHS_INFO

In [ ]:
SOURCE_EXTRACT_DIR = Path(CONFIG["source_extract_dir"])
MAT_FILES = []

files = find_source_mat_files(SOURCE_EXTRACT_DIR)

if files:
    MAT_FILES = files

if not MAT_FILES:
    raise FileNotFoundError(
        "Could not find Liu2024 source .mat files. "
    )

print(f"Source extract dir: {SOURCE_EXTRACT_DIR}")
print(f"Found {len(MAT_FILES)} .mat files")

# Write a structure preview for the first subject. This makes it clear whether
# the local files expose top-level rawdata/labels or an `eeg` struct.
preview_path = ARTIFACT_DIR / "mat_structure_preview_first_subject.csv"
mat_structure_preview(MAT_FILES[0]).to_csv(preview_path, index=False)
print(f"MAT structure preview saved to: {preview_path}")

subjects = []
for p in MAT_FILES:
    sid = subject_id_from_path(p)
    if CONFIG["subjects_to_use"] is not None and sid not in set(int(s) for s in CONFIG["subjects_to_use"]):
        continue
    if sid in set(int(s) for s in CONFIG["exclude_subjects"]):
        continue
    X_raw, y_raw, raw_field, label_field = load_subject_mat(p)
    validate_liu_source_subject(X_raw, y_raw, sid, path=p)
    subjects.append({
        "subject_id": sid,
        "path": str(p),
        "rawdata_shape": tuple(X_raw.shape),
        "labels_shape": tuple(y_raw.shape),
        "label_counts_raw": np.bincount(y_raw.astype(int), minlength=3).tolist(),
        "raw_field": raw_field,
        "label_field": label_field,
    })

subjects_df = pd.DataFrame(subjects).sort_values("subject_id").reset_index(drop=True)
if subjects_df.empty:
    raise RuntimeError("No subjects loaded.")

SUBJECTS = [int(s) for s in subjects_df["subject_id"].tolist()]
print(f"Subjects loaded: {SUBJECTS}")

subject_inventory_path = ARTIFACT_DIR / "subject_inventory.csv"
subjects_df.to_csv(subject_inventory_path, index=False)
print(f"Subject inventory saved to: {subject_inventory_path}")

EEG_INFO = make_liu_info(EFFECTIVE_SFREQ)
CHS_INFO = EEG_INFO["chs"]
CH_NAMES = list(EEG_CHANNEL_NAMES)

Xs, ys, subject_ids = [], [], []
window_summary_rows = []
preprocessing_steps_first_subject = None

for item in subjects_df.to_dict("records"):
    sid = int(item["subject_id"])
    X_raw, y_raw, _, _ = load_subject_mat(Path(item["path"]))
    X_win, y, samples_per_trial, preprocessing_steps, preprocessing_stats = preprocess_subject_configurable(X_raw, y_raw, sid)

    if preprocessing_steps_first_subject is None:
        preprocessing_steps_first_subject = preprocessing_steps

    Xs.append(X_win)
    ys.append(y)
    subject_ids.extend([sid] * len(y))
    window_summary_rows.append({
        "subject_id": sid,
        "n_windows": int(len(y)),
        "class_counts": np.bincount(y, minlength=TARGET_N_CLASSES).tolist(),
        "preprocessed_shape": tuple(X_win.shape),
        "resampled_samples_per_trial": int(samples_per_trial),
        "crop_start_sample": int(MI_WINDOW_START_SAMPLE),
        "crop_stop_sample": int(MI_WINDOW_STOP_SAMPLE),
        "target_window_samples": int(WINDOW_SAMPLES),
        "effective_window_duration_s": float(TARGET_TRIAL_DURATION_S),
        "source_unit": CONFIG["source_unit"],
        "final_model_unit": CONFIG["final_model_unit"],
        "demean_mode": CONFIG["demean_mode"],
        "reference_mode": CONFIG["reference_mode"],
        "reference_timing": CONFIG["reference_timing"],
        "resample": bool(CONFIG["resample"]),
        "effective_sfreq": float(EFFECTIVE_SFREQ),
        "filter_enabled": bool(CONFIG["filter_enabled"]),
        "filter_low": CONFIG.get("filter_low"),
        "filter_high": CONFIG.get("filter_high"),
        "filter_method": CONFIG.get("filter_method"),
        "mi_window_start_s": float(CONFIG["mi_window_start_s"]),
        "eog_correction": CONFIG.get("eog_correction"),
        "artifact_clip_mode": CONFIG.get("artifact_clip_mode"),
        "reject_bad_trials": bool(CONFIG.get("reject_bad_trials", False)),
        "normalization_mode": CONFIG.get("normalization_mode"),
        **preprocessing_stats,
    })

X_ALL = np.concatenate(Xs, axis=0)
Y_ALL = np.concatenate(ys, axis=0)
SUBJECT_ID_ALL = np.asarray(subject_ids)
print(f"X_ALL shape: {X_ALL.shape} | Y_ALL counts: {np.bincount(Y_ALL).tolist()}")

if preprocessing_steps_first_subject is not None:
    print("Preprocessing steps used:")
    for step in preprocessing_steps_first_subject:
        print(f"  - {step}")

window_summary_df = pd.DataFrame(window_summary_rows).sort_values("subject_id")
window_summary_path = ARTIFACT_DIR / "window_counts_by_subject.csv"
window_summary_df.to_csv(window_summary_path, index=False)
print(f"Window summary saved to: {window_summary_path}")
display(window_summary_df.head())

def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

SUBJECT_WINDOWS = {}
for sid in sorted(np.unique(SUBJECT_ID_ALL), key=_sort_subject_key):
    idx = np.where(SUBJECT_ID_ALL == sid)[0]
    SUBJECT_WINDOWS[str(sid)] = SubjectArrayDataset(X_ALL[idx], Y_ALL[idx], subject_id=sid)


# 4. Features: EA + tangent-at-identity, and S-JEPA embeddings

Euclidean Alignment whitens each subject's reference covariance toward identity, so the tangent
space **at the identity** is the right linearization — we vectorize `logm(C)` once per trial
(off-diagonals scaled by √2). The S-JEPA branch is loaded from the cached `.npz` if available.

In [ ]:
AD = CONFIG["adapt"]

def _inv_sqrt_spd(R, eps=1e-6):
    w, V = eigh(R); w = np.clip(w, eps, None); return (V * (1.0 / np.sqrt(w))) @ V.T

def _logm_spd(C, eps=1e-12):
    w, V = eigh(C); w = np.clip(w, eps, None); return (V * np.log(w)) @ V.T

def euclidean_align(X):
    covs = np.einsum("nct,ndt->ncd", X, X) / X.shape[2]
    P = _inv_sqrt_spd(covs.mean(0))
    return np.einsum("cd,ndt->nct", P, X)

def trial_covariances(X):
    if HAVE_PYRIEMANN:
        return Covariances(estimator=AD["cov_estimator"]).transform(X)
    n, c, t = X.shape; out = np.empty((n, c, c))
    for i in range(n):
        Xi = X[i] - X[i].mean(1, keepdims=True); C = Xi @ Xi.T / (t - 1)
        out[i] = C + 1e-3 * (np.trace(C) / c) * np.eye(c)
    return out

def tangent_at_identity(covs):
    """Vectorize logm(C) (upper triangle, off-diagonals * sqrt(2)). Reference = identity (EA)."""
    c = covs.shape[1]; iu = np.triu_indices(c); scale = np.sqrt(2) * np.ones((c, c)); np.fill_diagonal(scale, 1.0)
    out = np.empty((len(covs), len(iu[0])))
    for i, C in enumerate(covs):
        L = _logm_spd(C) * scale; out[i] = L[iu]
    return out

def get_submodule_by_suffix(model, name):
    for n, m in model.named_modules():
        if n == name or n.endswith("." + name): return m
    return None

def extract_embeddings(model, X, device, module_name, pool, batch):
    model.eval().to(device)
    target = get_submodule_by_suffix(model, module_name)
    if target is None: raise RuntimeError(f"submodule '{module_name}' not found")
    store = {}; h = target.register_forward_pre_hook(lambda m, inp: store.__setitem__("z", inp[0].detach()))
    out = []
    try:
        with torch.no_grad():
            for i in range(0, len(X), batch):
                xb = torch.as_tensor(np.asarray(X[i:i+batch]), dtype=torch.float32, device=device)
                model(xb); z = store["z"]
                if z.ndim == 3: z = z.mean(1) if pool == "mean" else z.reshape(z.shape[0], -1)
                elif z.ndim > 3: z = z.reshape(z.shape[0], -1)
                out.append(z.float().cpu().numpy())
    finally:
        h.remove()
    return np.concatenate(out, 0)


## 5. Build aligned feature matrices (computed once)

In [ ]:
SUBJ = np.array([str(s) for s in SUBJECT_ID_ALL]); Y = Y_ALL.astype(int)
subjects_all = sorted(set(SUBJ), key=lambda s: int(s))

X_aligned = np.empty_like(X_ALL, dtype=np.float64)
for sid in subjects_all:
    m = SUBJ == sid; Xs = X_ALL[m].astype(np.float64)
    X_aligned[m] = euclidean_align(Xs) if AD["euclidean_alignment"] else Xs

TAN_ALL = tangent_at_identity(trial_covariances(X_aligned))
print(f"Tangent features: {TAN_ALL.shape}")

EMB_ALL, SJEPA_OK = None, False
p = AD.get("sjepa_embeddings_path")
if p and Path(p).exists():
    EMB_ALL = np.load(p)["X"]; SJEPA_OK = EMB_ALL.shape[0] == len(Y)
    print(f"Loaded cached embeddings {EMB_ALL.shape}: ok={SJEPA_OK}")
elif HAVE_TORCH and HAVE_BRAINDECODE:
    try:
        model, _ = build_model()
        src = X_aligned if AD["align_before_sjepa"] else X_ALL.astype(np.float64)
        EMB_ALL = extract_embeddings(model, src, DEVICE, AD["embedding_module"], AD["embedding_pool"], AD["embedding_batch"])
        SJEPA_OK = True; np.savez(ARTIFACT_DIR / "sjepa_embeddings.npz", X=EMB_ALL)
        print(f"Extracted embeddings {EMB_ALL.shape}.")
    except Exception as exc:
        print(f"[S-JEPA] extraction failed -> riemann-only. {exc}")
else:
    print("[S-JEPA] no cached embeddings + no torch -> riemann-only. Set adapt['sjepa_embeddings_path'].")

branches = [b for b in AD["branches"] if not (b in ("sjepa", "fusion") and not SJEPA_OK)]
strategies = AD["strategies"]
print("branches:", branches, "| strategies:", strategies)

def branch_matrix(branch):
    if branch == "riemann": return TAN_ALL
    if branch == "sjepa":   return EMB_ALL
    return np.concatenate([TAN_ALL, EMB_ALL], axis=1)  # fusion


# 6. Calibration-transfer evaluation

For a target subject, a calibration size `K`, and a random draw: split the target's trials into
`K` stratified calibration trials and the rest as test. Train each strategy's head on the
appropriate set and score the (shared) test trials. Fold-safe: scaler fit on the training rows.

In [ ]:
def _fit_predict(F, ytr, tr_rows, te_rows, sample_weight=None):
    Xtr, Xte = F[tr_rows], F[te_rows]
    if AD["standardize"]:
        sc = StandardScaler().fit(Xtr); Xtr, Xte = sc.transform(Xtr), sc.transform(Xte)
    clf = LogisticRegression(max_iter=2000, C=AD["logreg_C"])
    clf.fit(Xtr, ytr, sample_weight=sample_weight)
    return clf.predict(Xte)

def stratified_calibration(target_rows, y, K, rng):
    if K <= 0:
        return np.array([], dtype=int), target_rows.copy()
    cal = []
    for c in np.unique(y[target_rows]):
        ci = target_rows[y[target_rows] == c]; rng.shuffle(ci)
        cal += list(ci[:max(K // 2, 1)])
    cal = np.array(cal, dtype=int)
    test = np.array([i for i in target_rows if i not in set(cal)], dtype=int)
    return cal, test

def run_calibration_curve(target, branch):
    F = branch_matrix(branch)
    target_rows = np.where(SUBJ == str(target))[0]
    pool_rows = np.where(SUBJ != str(target))[0]
    rows = []
    for K in AD["calibration_grid"]:
        for rep in range(AD["n_repeats"]):
            rng = np.random.default_rng(AD["seed"] + 7919 * int(target) + 31 * K + rep)
            cal, test = stratified_calibration(target_rows, Y, K, rng)
            if len(test) == 0 or len(np.unique(Y[test])) < 2:
                continue
            for strat in strategies:
                if strat == "within_only":
                    if len(cal) == 0 or len(np.unique(Y[cal])) < 2:
                        continue
                    yp = _fit_predict(F, Y[cal], cal, test)
                elif strat == "transfer_only":
                    yp = _fit_predict(F, Y[pool_rows], pool_rows, test)
                else:  # transfer_plus_cal
                    tr = np.concatenate([pool_rows, cal]) if len(cal) else pool_rows
                    sw = None
                    if len(cal) and AD["calibration_upweight"] != 1.0:
                        sw = np.concatenate([np.ones(len(pool_rows)),
                                             np.full(len(cal), AD["calibration_upweight"])])
                    yp = _fit_predict(F, Y[tr], tr, test, sample_weight=sw)
                rows.append({"target": str(target), "branch": branch, "strategy": strat,
                             "K": K, "rep": rep, "balanced_accuracy": balanced_accuracy_score(Y[test], yp)})
    return rows


# 7. Run

In [ ]:
targets = (subjects_all if AD["target_subjects"] == "all"
           else [str(s) for s in AD["target_subjects"] if str(s) in set(SUBJ)])
print(f"Targets ({len(targets)}): {targets}\n")

ALL = []
for t in targets:
    for b in branches:
        ALL.extend(run_calibration_curve(t, b))
RES = pd.DataFrame(ALL)
RES.to_csv(ARTIFACT_DIR / "calibration_transfer_results.csv", index=False)

# mean over subjects+reps: curve[branch][strategy] vs K
curve = RES.groupby(["branch", "strategy", "K"])["balanced_accuracy"].mean().reset_index()
curve.to_csv(ARTIFACT_DIR / "calibration_curve.csv", index=False)
print("Mean balanced accuracy by branch / strategy / K:")
for b in branches:
    print(f"\n[{b}]")
    sub = curve[curve["branch"] == b].pivot(index="K", columns="strategy", values="balanced_accuracy")
    print((100 * sub).round(1).to_string())

# headline at the largest K
Kmax = max(AD["calibration_grid"])
head = curve[curve["K"] == Kmax].pivot(index="branch", columns="strategy", values="balanced_accuracy")
summary = {"targets": targets, "K_max": Kmax,
           "headline_at_Kmax": (100 * head).round(1).to_dict(),
           "branches": branches, "strategies": strategies}
with open(ARTIFACT_DIR / "calibration_summary.json", "w") as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n================ at K={Kmax} (balanced accuracy %) ================")
print((100 * head).round(1).to_string())


# 8. Calibration curves

In [ ]:
if HAVE_MPL and not RES.empty:
    fig, axes = plt.subplots(1, len(branches), figsize=(5 * len(branches), 4.2), squeeze=False)
    for j, b in enumerate(branches):
        ax = axes[0][j]
        for strat in strategies:
            s = curve[(curve["branch"] == b) & (curve["strategy"] == strat)].sort_values("K")
            if strat == "within_only":
                s = s[s["K"] > 0]
            ax.plot(s["K"], s["balanced_accuracy"], "o-", label=strat)
        ax.axhline(0.5, ls="--", c="grey", lw=1)
        ax.axhline(0.7, ls=":", c="green", lw=1)
        ax.set_title(f"branch: {b}"); ax.set_xlabel("calibration trials K"); ax.set_ylim(0.4, 0.85)
        if j == 0: ax.set_ylabel("balanced accuracy"); 
        ax.legend(fontsize=8)
    fig.suptitle("Subject-adaptive calibration curves (mean over decodable targets)")
    fig.tight_layout(); fig.savefig(ARTIFACT_DIR / "calibration_curves.png", dpi=160); plt.close(fig)
    print(f"Saved: {ARTIFACT_DIR/'calibration_curves.png'}")
else:
    print("Plots skipped.")


## 9. Save run metadata

In [ ]:
run_metadata = {
    "run_id": RUN_ID, "artifact_dir": str(ARTIFACT_DIR),
    "experiment_name": CONFIG["experiment_name"], "config_note": CONFIG["config_note"],
    "adapt_config": AD, "branches_run": branches, "sjepa_available": bool(SJEPA_OK),
    "tangent_backend": "pyriemann" if HAVE_PYRIEMANN else "numpy_fallback",
    "summary": summary,
}
with open(ARTIFACT_DIR / "run_metadata.json", "w") as f:
    json.dump(run_metadata, f, indent=2, default=str)
print(f"Saved run metadata to: {ARTIFACT_DIR/'run_metadata.json'}")
for pth in sorted(ARTIFACT_DIR.glob("*")):
    print(f"  - {pth.name}")
try:
    _LOG_FILE_HANDLE.close()
except Exception:
    pass


# 10. How to read this

- **Compare the three strategies within each branch.** `transfer_only` is flat in `K` (it ignores
  calibration) and — from the LOSO result — should sit near chance. `within_only` should rise
  with `K` toward the ~75% within-subject ceiling. `transfer_plus_cal` is the test of value:
  - if it **beats** `within_only` at small `K`, cross-subject pretraining genuinely reduces the
    calibration burden — a deployable, reportable win;
  - if it **overlaps** `within_only`, only the subject's own trials matter and transfer adds
    nothing (the honest likely outcome given the chance LOSO result).
- **Compare branches.** If `fusion` ≈ `riemann` and `sjepa` trails, S-JEPA isn't contributing
  beyond the Riemannian features — quantified, not hand-waved.
- **Knobs:** `calibration_upweight` (weight calibration trials more in `transfer_plus_cal`),
  `target_subjects` (try `"all"` to confirm non-decodable subjects stay at chance regardless of
  `K`), `logreg_C`, and `align_before_sjepa`.

**Honest framing for the writeup.** Any ≥70% here at larger `K` is within-subject Riemannian
accuracy; the contribution of transfer and of S-JEPA is exactly the *gap* between
`transfer_plus_cal`/`fusion` and `within_only`/`riemann`. Reporting that gap — even if it is
~0 — is the scientific result: it states precisely what cross-subject pretraining and the SSL
embedding do and do not buy on this dataset.